<a href="https://colab.research.google.com/github/Wuanzz/FER-Video-Emotion-Recognition/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Kết nối Google Drive & Khởi tạo Cấu hình Đường dẫn Dự án

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video'
DATASET_ZIP = os.path.join(PROJECT_DIR, 'ravdess.zip')

PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_data')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("ZIP:", DATASET_ZIP)
print("Tồn tại:", os.path.exists(DATASET_ZIP))

if os.path.exists(DATASET_ZIP):
    print("Dung lượng ZIP:",
          round(os.path.getsize(DATASET_ZIP) / (1024**3), 2),
          "GB")

Mounted at /content/drive
ZIP: /content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video/ravdess.zip
Tồn tại: True
Dung lượng ZIP: 23.86 GB


# Cell 2: Kiểm tra & Phân loại Tập tin Dữ liệu trong File Nén ZIP

In [ ]:
import zipfile
import os

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    files = z.namelist()

print("Tổng số file trong ZIP:", len(files))

mp4_files = [
    f for f in files
    if f.lower().endswith('.mp4')
]

wav_files = [
    f for f in files
    if f.lower().endswith('.wav')
]

print("Số video MP4:", len(mp4_files))
print("Số audio WAV:", len(wav_files))

print("\nMột số video:")
for f in mp4_files[:10]:
    print(f)

Tổng số file trong ZIP: 7356
Số video MP4: 4904
Số audio WAV: 2452

Một số video:
Video_Song_Actor_01/Actor_01/01-02-01-01-01-01-01.mp4
Video_Song_Actor_01/Actor_01/01-02-01-01-01-02-01.mp4
Video_Song_Actor_01/Actor_01/01-02-01-01-02-01-01.mp4
Video_Song_Actor_01/Actor_01/01-02-01-01-02-02-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-01-01-01-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-01-01-02-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-01-02-01-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-01-02-02-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-02-01-01-01.mp4
Video_Song_Actor_01/Actor_01/01-02-02-02-01-02-01.mp4


# Cell 3: Cấu hình Môi trường Thư viện OpenCV Tương thích

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install -q "opencv-python-headless==4.13.0.92"

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-contrib-python 4.14.0.94
Uninstalling opencv-contrib-python-4.14.0.94:
  Successfully uninstalled opencv-contrib-python-4.14.0.94
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 15.8 MB/s eta 0:00:00


# Cell 4: Kiểm tra Phiên bản Thư viện & Khởi tạo Face Detector (Haar Cascade)

In [ ]:
import cv2
import numpy as np

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("CascadeClassifier:", hasattr(cv2, 'CascadeClassifier'))

cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("Không thể load Haar Cascade!")

print("Face detector: OK")

NumPy: 2.1.3
OpenCV: 4.13.0
CascadeClassifier: True
Face detector: OK


# Cell 5: Thiết lập Hằng số Tiền xử lý & Ánh xạ Nhãn Cảm xúc

In [ ]:
IMAGE_SIZE = (224, 224)
SEQUENCE_LENGTH = 16

EMOTION_MAP = {
    1: 'neutral',
    2: 'calm',
    3: 'happy',
    4: 'sad',
    5: 'angry',
    6: 'fearful',
    7: 'disgust',
    8: 'surprised'
}

print("IMAGE_SIZE:", IMAGE_SIZE)
print("SEQUENCE_LENGTH:", SEQUENCE_LENGTH)
print("Số emotion:", len(EMOTION_MAP))

IMAGE_SIZE: (224, 224)
SEQUENCE_LENGTH: 16
Số emotion: 8


# Cell 6: Định nghĩa Hàm Trích xuất & Tiền xử lý Chuỗi Khung hình Mặt từ Video

In [ ]:
def process_video(video_path, sequence_length=16, image_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Không mở được video: {video_path}")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        sequence_length,
        dtype=int
    )

    frames = []

    for frame_idx in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        success, frame = cap.read()

        if not success or frame is None:
            if len(frames) > 0:
                frames.append(frames[-1].copy())
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        if len(faces) > 0:

            x, y, w, h = max(
                faces,
                key=lambda rect: rect[2] * rect[3]
            )

            pad_x = int(0.1 * w)
            pad_y = int(0.1 * h)

            x1 = max(0, x - pad_x)
            y1 = max(0, y - pad_y)
            x2 = min(frame.shape[1], x + w + pad_x)
            y2 = min(frame.shape[0], y + h + pad_y)

            face = frame[y1:y2, x1:x2]

        else:
            face = frame

        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, image_size)

        frames.append(face)

    cap.release()

    if len(frames) == 0:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    frames = frames[:sequence_length]

    return np.array(frames, dtype=np.uint8)

# Cell 7: Trích xuất Dữ liệu Mẫu (Sample Data - 20 Video) cho Nhóm

In [ ]:
import zipfile
import os

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:

    mp4_files = [
        f for f in z.namelist()
        if f.lower().endswith('.mp4')
        and f.startswith('Video_Speech_Actor_')
    ]

print("Tổng video Speech:", len(mp4_files))

sample_files = mp4_files[:20]

X_sample = []
y_sample = []

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:

    for i, member in enumerate(sample_files):

        filename = os.path.basename(member)
        parts = filename.replace('.mp4', '').split('-')

        if len(parts) < 3:
            print("Tên file không hợp lệ:", filename)
            continue

        try:
            emotion_code = int(parts[2])
        except ValueError:
            continue

        if emotion_code not in EMOTION_MAP:
            continue

        label = emotion_code - 1

        temp_path = '/content/temp_ravdess_video.mp4'

        # Giải nén tạm 1 video
        with z.open(member) as source, open(temp_path, 'wb') as target:
            target.write(source.read())

        # Xử lý video
        sequence = process_video(
            temp_path,
            sequence_length=SEQUENCE_LENGTH,
            image_size=IMAGE_SIZE
        )

        # Xóa video tạm
        os.remove(temp_path)

        if sequence is not None:
            X_sample.append(sequence)
            y_sample.append(label)

            print(
                f"[{i+1}/{len(sample_files)}] "
                f"{filename} → "
                f"{EMOTION_MAP[emotion_code]} → "
                f"{sequence.shape}"
            )

X_sample = np.array(X_sample, dtype=np.uint8)
y_sample = np.array(y_sample, dtype=np.int64)

print("\n===== SAMPLE SPEECH VIDEO =====")
print("X shape:", X_sample.shape)
print("y shape:", y_sample.shape)
print("X dtype:", X_sample.dtype)
print("y:", y_sample)

Tổng video Speech: 2880
[1/20] 01-01-01-01-01-01-01.mp4 → neutral → (16, 224, 224, 3)
[2/20] 01-01-01-01-01-02-01.mp4 → neutral → (16, 224, 224, 3)
[3/20] 01-01-01-01-02-01-01.mp4 → neutral → (16, 224, 224, 3)
[4/20] 01-01-01-01-02-02-01.mp4 → neutral → (16, 224, 224, 3)
[5/20] 01-01-02-01-01-01-01.mp4 → calm → (16, 224, 224, 3)
[6/20] 01-01-02-01-01-02-01.mp4 → calm → (16, 224, 224, 3)
[7/20] 01-01-02-01-02-01-01.mp4 → calm → (16, 224, 224, 3)
[8/20] 01-01-02-01-02-02-01.mp4 → calm → (16, 224, 224, 3)
[9/20] 01-01-02-02-01-01-01.mp4 → calm → (16, 224, 224, 3)
[10/20] 01-01-02-02-01-02-01.mp4 → calm → (16, 224, 224, 3)
[11/20] 01-01-02-02-02-01-01.mp4 → calm → (16, 224, 224, 3)
[12/20] 01-01-02-02-02-02-01.mp4 → calm → (16, 224, 224, 3)
[13/20] 01-01-03-01-01-01-01.mp4 → happy → (16, 224, 224, 3)
[14/20] 01-01-03-01-01-02-01.mp4 → happy → (16, 224, 224, 3)
[15/20] 01-01-03-01-02-01-01.mp4 → happy → (16, 224, 224, 3)
[16/20] 01-01-03-01-02-02-01.mp4 → happy → (16, 224, 224, 3)
[17/20] 0

# Cell 8: Tiền xử lý Toàn bộ Dataset theo Batch & Cơ chế Khôi phục

In [ ]:
import zipfile
import os
import numpy as np
import time

total_videos = len(speech_videos)
num_batches = int(np.ceil(total_videos / BATCH_SIZE))

overall_start = time.time()

print("=" * 60)
print("RESUME PREPROCESSING RAVDESS")
print("=" * 60)
print("Tổng video:", total_videos)
print("Tổng batch:", num_batches)
print()

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:

    for batch_idx in range(num_batches):

        batch_number = batch_idx + 1

        batch_path = os.path.join(
            BATCH_DIR,
            f'batch_{batch_number:03d}.npz'
        )

        # Nếu batch đã tồn tại → bỏ qua
        if os.path.exists(batch_path):

            print(
                f"[SKIP] Batch {batch_number:03d} "
                f"đã tồn tại."
            )

            continue

        batch_start = batch_idx * BATCH_SIZE
        batch_end = min(
            batch_start + BATCH_SIZE,
            total_videos
        )

        batch_files = speech_videos[
            batch_start:batch_end
        ]

        batch_X = []
        batch_y = []

        batch_start_time = time.time()

        print()
        print(
            f"===== BATCH {batch_number:03d}/{num_batches} "
            f"({batch_start + 1}-{batch_end}) ====="
        )

        for local_idx, member in enumerate(batch_files):

            filename = os.path.basename(member)

            parts = filename.replace(
                '.mp4', ''
            ).split('-')

            if len(parts) < 3:
                continue

            try:
                emotion_code = int(parts[2])
            except ValueError:
                continue

            if emotion_code not in EMOTION_MAP:
                continue

            label = emotion_code - 1

            temp_path = '/content/temp_ravdess_video.mp4'

            try:

                # Extract tạm video
                with z.open(member) as source:
                    with open(temp_path, 'wb') as target:
                        target.write(source.read())

                # Preprocess
                sequence = process_video(
                    temp_path,
                    sequence_length=SEQUENCE_LENGTH,
                    image_size=IMAGE_SIZE
                )

                if sequence is not None:
                    batch_X.append(sequence)
                    batch_y.append(label)

            except Exception as e:

                print(
                    f"Lỗi video {filename}: {e}"
                )

            finally:

                if os.path.exists(temp_path):
                    os.remove(temp_path)

            current = batch_start + local_idx + 1

            if (local_idx + 1) % 10 == 0:

                print(
                    f"  Đã xử lý: "
                    f"{current}/{total_videos}"
                )

        # Convert NumPy
        batch_X = np.array(
            batch_X,
            dtype=np.uint8
        )

        batch_y = np.array(
            batch_y,
            dtype=np.int64
        )

        # Lưu batch
        np.savez_compressed(
            batch_path,
            X=batch_X,
            y=batch_y
        )

        elapsed = time.time() - batch_start_time

        print()
        print(
            f"Batch {batch_number:03d} HOÀN THÀNH"
        )
        print(
            f"X shape: {batch_X.shape}"
        )
        print(
            f"y shape: {batch_y.shape}"
        )
        print(
            f"Thời gian: {elapsed / 60:.2f} phút"
        )
        print(
            f"Đã lưu: {batch_path}"
        )

total_time = time.time() - overall_start

print()
print("=" * 60)
print("RESUME HOÀN TẤT")
print("=" * 60)
print(
    f"Tổng thời gian phiên này: "
    f"{total_time / 60:.2f} phút"
)

RESUME PREPROCESSING RAVDESS
Tổng video: 2880
Tổng batch: 29

[SKIP] Batch 001 đã tồn tại.
[SKIP] Batch 002 đã tồn tại.
[SKIP] Batch 003 đã tồn tại.
[SKIP] Batch 004 đã tồn tại.
[SKIP] Batch 005 đã tồn tại.
[SKIP] Batch 006 đã tồn tại.
[SKIP] Batch 007 đã tồn tại.
[SKIP] Batch 008 đã tồn tại.
[SKIP] Batch 009 đã tồn tại.

===== BATCH 010/29 (901-1000) =====
  Đã xử lý: 910/2880
  Đã xử lý: 920/2880
  Đã xử lý: 930/2880
  Đã xử lý: 940/2880
  Đã xử lý: 950/2880
  Đã xử lý: 960/2880
  Đã xử lý: 970/2880
  Đã xử lý: 980/2880
  Đã xử lý: 990/2880
  Đã xử lý: 1000/2880

Batch 010 HOÀN THÀNH
X shape: (100, 16, 224, 224, 3)
y shape: (100,)
Thời gian: 8.94 phút
Đã lưu: /content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video/processed_data/ravdess_batches/batch_010.npz

===== BATCH 011/29 (1001-1100) =====
  Đã xử lý: 1010/2880
  Đã xử lý: 1020/2880
  Đã xử lý: 1030/2880
  Đã xử lý: 1040/2880
  Đã xử lý: 1050/2880
  Đã xử lý: 1060/2880
  Đã xử lý: 1070/2880
  Đã xử lý: 1080/2880
  

# Cell 9: Kiểm tra Toàn vẹn Dữ liệu

In [11]:
import os
import glob
import numpy as np

batch_files = sorted(
    glob.glob(
        os.path.join(BATCH_DIR, 'batch_*.npz')
    )
)

print("===== DATASET VALIDATION =====")

print("Số batch:", len(batch_files))

total_samples = 0
all_labels = []

for path in batch_files:

    data = np.load(path)

    X_batch = data['X']
    y_batch = data['y']

    print(
        f"{os.path.basename(path)}: "
        f"X={X_batch.shape}, "
        f"y={y_batch.shape}"
    )

    # Kiểm tra shape
    assert X_batch.ndim == 5
    assert X_batch.shape[1:] == (16, 224, 224, 3)

    # Kiểm tra số lượng
    assert len(X_batch) == len(y_batch)

    # Kiểm tra NaN
    assert not np.isnan(X_batch.astype(np.float32)).any()

    total_samples += len(y_batch)
    all_labels.extend(y_batch.tolist())

all_labels = np.array(all_labels)

print("\n===== SUMMARY =====")
print("Tổng samples:", total_samples)
print("X shape mỗi video:", (16, 224, 224, 3))
print("Số classes:", len(np.unique(all_labels)))

print("\n===== LABEL DISTRIBUTION =====")

unique, counts = np.unique(
    all_labels,
    return_counts=True
)

for label, count in zip(unique, counts):
    emotion_name = EMOTION_MAP[label + 1]
    print(
        f"{label}: "
        f"{emotion_name:10s} → {count}"
    )

assert total_samples == 2880
assert len(unique) == 8

print("\n✓ DATASET VALIDATION PASSED")

===== DATASET VALIDATION =====
Số batch: 29
batch_001.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_002.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_003.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_004.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_005.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_006.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_007.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_008.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_009.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_010.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_011.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_012.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_013.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_014.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_015.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_016.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_017.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_018.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_019.npz: X=(100, 16, 224, 224, 3), y=(100,)
batch_